In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read from bronze table

In [0]:
df = spark.table("workspace.silver.crm_customers")

df.display()

In [0]:
df.printSchema()

### Gold transformations & write into Gold table

Join tables on customer_id

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_customer
USING DELTA
AS
SELECT
    a.customer_id,
    a.first_name,
    a.last_name,
    b.birthdate,
      FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), b.birthdate) / 12) AS age,
    a.marital_status,
    a.gender,
    c.country
FROM workspace.silver.crm_customers AS a
LEFT JOIN workspace.silver.erp_customers AS b
    ON a.customer_id = b.customer_id
LEFT JOIN workspace.silver.erp_locations AS c
    ON b.customer_id = c.customer_id;

### Write into Gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customer")